In [0]:
# app.py
# ==============================================================================
# Aplicación Streamlit para Dashboard de Segmentación RFM
# Consume datos desde la Capa Gold en Databricks
# ==============================================================================

import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from databricks import sql
import os

# ==============================================================================
# CONFIGURACIÓN DE LA PÁGINA
# ==============================================================================

st.set_page_config(
    page_title="Dashboard RFM - Segmentación de Clientes",
    page_icon="📊",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ==============================================================================
# CONEXIÓN A DATABRICKS
# ==============================================================================

@st.cache_resource
def get_databricks_connection():
    """
    Establece conexión con Databricks SQL Warehouse
    NOTA: Configurar las credenciales como secrets en Streamlit
    """
    try:
        connection = sql.connect(
            server_hostname=os.getenv("DATABRICKS_SERVER_HOSTNAME"),
            http_path=os.getenv("DATABRICKS_HTTP_PATH"),
            access_token=os.getenv("DATABRICKS_TOKEN")
        )
        return connection
    except Exception as e:
        st.error(f"Error conectando a Databricks: {e}")
        st.info("Configurar variables de entorno: DATABRICKS_SERVER_HOSTNAME, DATABRICKS_HTTP_PATH, DATABRICKS_TOKEN")
        return None

# ==============================================================================
# FUNCIONES DE CARGA DE DATOS
# ==============================================================================

@st.cache_data(ttl=300)  # Cache por 5 minutos
def load_customer_segments():
    """Carga la tabla de segmentos de clientes desde Gold"""
    conn = get_databricks_connection()
    
    if conn is None:
        return None
    
    query = """
        SELECT 
            customer_unique_id,
            Segmento,
            Segmento_Nombre,
            Recencia,
            Frecuencia,
            Monetario,
            Amplitud_Categorias,
            Total_Articulos,
            R_Score,
            F_Score,
            M_Score,
            RFM_Score
        FROM gold.customer_segments
    """
    
    with conn.cursor() as cursor:
        cursor.execute(query)
        result = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]
    
    df = pd.DataFrame(result, columns=columns)
    return df


@st.cache_data(ttl=300)
def load_segment_profiles():
    """Carga el perfil agregado de cada segmento desde Gold"""
    conn = get_databricks_connection()
    
    if conn is None:
        return None
    
    query = """
        SELECT 
            Segmento,
            Segmento_Nombre,
            Customer_Count,
            Avg_Recencia,
            Avg_Frecuencia,
            Avg_Monetario,
            Total_Revenue,
            Pct_Customers,
            Pct_Revenue,
            Value_Index
        FROM gold.segment_profiles
        ORDER BY Total_Revenue DESC
    """
    
    with conn.cursor() as cursor:
        cursor.execute(query)
        result = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]
    
    df = pd.DataFrame(result, columns=columns)
    return df

# ==============================================================================
# INTERFAZ PRINCIPAL
# ==============================================================================

# Título principal
st.title("📊 Dashboard de Segmentación RFM")
st.markdown("---")

# Sidebar
with st.sidebar:
    st.header("⚙️ Configuración")
    
    st.markdown("### Acerca de")
    st.info("""
    Este dashboard muestra la segmentación de clientes basada en análisis RFM 
    (Recency, Frequency, Monetary) utilizando K-Means clustering.
    
    **Datos desde**: Databricks Gold Layer  
    **Actualización**: Cada 5 minutos
    """)
    
    if st.button("🔄 Refrescar Datos"):
        st.cache_data.clear()
        st.rerun()

# ==============================================================================
# CARGAR DATOS
# ==============================================================================

with st.spinner("Cargando datos desde Databricks..."):
    df_customers = load_customer_segments()
    df_profiles = load_segment_profiles()

if df_customers is None or df_profiles is None:
    st.error("❌ No se pudieron cargar los datos. Verifica la conexión a Databricks.")
    st.stop()

# ==============================================================================
# SECCIÓN 1: MÉTRICAS CLAVE
# ==============================================================================

st.header("📈 Métricas Clave")

col1, col2, col3, col4 = st.columns(4)

with col1:
    total_customers = len(df_customers)
    st.metric("Total Clientes", f"{total_customers:,}")

with col2:
    total_revenue = df_profiles['Total_Revenue'].sum()
    st.metric("Ingresos Totales", f"R$ {total_revenue:,.2f}")

with col3:
    avg_ticket = df_customers['Monetario'].mean()
    st.metric("Ticket Promedio", f"R$ {avg_ticket:,.2f}")

with col4:
    num_segments = len(df_profiles)
    st.metric("Segmentos", num_segments)

st.markdown("---")

# ==============================================================================
# SECCIÓN 2: DISTRIBUCIÓN DE SEGMENTOS
# ==============================================================================

st.header("🎯 Distribución de Segmentos")

col1, col2 = st.columns(2)

with col1:
    # Gráfico de pastel - Clientes por segmento
    fig_customers = px.pie(
        df_profiles,
        values='Customer_Count',
        names='Segmento_Nombre',
        title='Distribución de Clientes por Segmento',
        hole=0.4,
        color_discrete_sequence=px.colors.qualitative.Set3
    )
    fig_customers.update_traces(textposition='inside', textinfo='percent+label')
    st.plotly_chart(fig_customers, use_container_width=True)

with col2:
    # Gráfico de pastel - Ingresos por segmento
    fig_revenue = px.pie(
        df_profiles,
        values='Total_Revenue',
        names='Segmento_Nombre',
        title='Distribución de Ingresos por Segmento',
        hole=0.4,
        color_discrete_sequence=px.colors.qualitative.Pastel
    )
    fig_revenue.update_traces(textposition='inside', textinfo='percent+label')
    st.plotly_chart(fig_revenue, use_container_width=True)

st.markdown("---")

# ==============================================================================
# SECCIÓN 3: TABLA DE PERFILES DE SEGMENTOS
# ==============================================================================

st.header("📊 Perfil Detallado de Segmentos")

# Preparar tabla para mostrar
display_profiles = df_profiles[[
    'Segmento_Nombre', 
    'Customer_Count', 
    'Pct_Customers',
    'Total_Revenue',
    'Pct_Revenue', 
    'Value_Index',
    'Avg_Monetario',
    'Avg_Frecuencia',
    'Avg_Recencia'
]].copy()

# Formatear columnas
display_profiles['Pct_Customers'] = display_profiles['Pct_Customers'].apply(lambda x: f"{x:.1f}%")
display_profiles['Pct_Revenue'] = display_profiles['Pct_Revenue'].apply(lambda x: f"{x:.1f}%")
display_profiles['Total_Revenue'] = display_profiles['Total_Revenue'].apply(lambda x: f"R$ {x:,.2f}")
display_profiles['Avg_Monetario'] = display_profiles['Avg_Monetario'].apply(lambda x: f"R$ {x:,.2f}")
display_profiles['Value_Index'] = display_profiles['Value_Index'].apply(lambda x: f"{x:.2f}x")
display_profiles['Avg_Frecuencia'] = display_profiles['Avg_Frecuencia'].apply(lambda x: f"{x:.2f}")
display_profiles['Avg_Recencia'] = display_profiles['Avg_Recencia'].apply(lambda x: f"{x:.0f} días")

# Renombrar columnas para visualización
display_profiles.columns = [
    'Segmento', 
    '# Clientes', 
    '% Clientes',
    'Ingresos Totales',
    '% Ingresos', 
    'Índice de Valor',
    'Gasto Promedio',
    'Frecuencia Promedio',
    'Recencia Promedio'
]

st.dataframe(display_profiles, use_container_width=True, hide_index=True)

st.markdown("---")

# ==============================================================================
# SECCIÓN 4: DISTRIBUCIONES RFM
# ==============================================================================

st.header("📉 Distribuciones RFM")

tab1, tab2, tab3 = st.tabs(["Recencia", "Frecuencia", "Monetario"])

with tab1:
    # Histograma de Recencia
    fig_recencia = px.histogram(
        df_customers,
        x='Recencia',
        color='Segmento_Nombre',
        nbins=50,
        title='Distribución de Recencia (Días desde última compra)',
        labels={'Recencia': 'Días', 'count': 'Número de Clientes'},
        color_discrete_sequence=px.colors.qualitative.Set3
    )
    fig_recencia.update_layout(barmode='overlay')
    fig_recencia.update_traces(opacity=0.7)
    st.plotly_chart(fig_recencia, use_container_width=True)
    
    st.info("**Recencia**: Días transcurridos desde la última compra. Valores bajos = Clientes activos.")

with tab2:
    # Histograma de Frecuencia (escala logarítmica por sesgo)
    fig_frecuencia = px.histogram(
        df_customers,
        x='Frecuencia',
        color='Segmento_Nombre',
        title='Distribución de Frecuencia (Número de compras)',
        labels={'Frecuencia': 'Número de Compras', 'count': 'Número de Clientes'},
        color_discrete_sequence=px.colors.qualitative.Set3,
        log_y=True  # Escala logarítmica por distribución sesgada
    )
    st.plotly_chart(fig_frecuencia, use_container_width=True)
    
    # Mostrar estadística clave
    single_purchase_pct = (df_customers['Frecuencia'] == 1).sum() / len(df_customers) * 100
    st.warning(f"⚠️ **{single_purchase_pct:.1f}%** de clientes tienen solo 1 compra")

with tab3:
    # Box plot de Monetario por segmento
    fig_monetario = px.box(
        df_customers,
        x='Segmento_Nombre',
        y='Monetario',
        color='Segmento_Nombre',
        title='Distribución de Valor Monetario por Segmento',
        labels={'Monetario': 'Gasto Total (R$)', 'Segmento_Nombre': 'Segmento'},
        color_discrete_sequence=px.colors.qualitative.Set3
    )
    fig_monetario.update_layout(showlegend=False)
    st.plotly_chart(fig_monetario, use_container_width=True)
    
    st.info("**Monetario**: Gasto total acumulado del cliente. Identifica clientes de alto valor.")

st.markdown("---")

# ==============================================================================
# SECCIÓN 5: EXPLORADOR DE CLIENTES
# ==============================================================================

st.header("🔍 Explorador de Clientes")

# Filtros
col1, col2 = st.columns(2)

with col1:
    selected_segments = st.multiselect(
        "Filtrar por Segmento",
        options=df_customers['Segmento_Nombre'].unique(),
        default=df_customers['Segmento_Nombre'].unique()
    )

with col2:
    min_monetary = st.number_input(
        "Gasto Mínimo (R$)",
        min_value=0.0,
        max_value=float(df_customers['Monetario'].max()),
        value=0.0,
        step=100.0
    )

# Aplicar filtros
filtered_customers = df_customers[
    (df_customers['Segmento_Nombre'].isin(selected_segments)) &
    (df_customers['Monetario'] >= min_monetary)
]

st.write(f"Mostrando {len(filtered_customers):,} clientes")

# Tabla de clientes
display_customers = filtered_customers[[
    'customer_unique_id',
    'Segmento_Nombre',
    'RFM_Score',
    'Recencia',
    'Frecuencia',
    'Monetario'
]].copy()

display_customers.columns = [
    'ID Cliente',
    'Segmento',
    'RFM Score',
    'Recencia (días)',
    'Frecuencia',
    'Monetario (R$)'
]

st.dataframe(
    display_customers.head(100),  # Mostrar primeros 100
    use_container_width=True,
    hide_index=True
)

# Botón de descarga
csv = filtered_customers.to_csv(index=False)
st.download_button(
    label="📥 Descargar datos filtrados (CSV)",
    data=csv,
    file_name="clientes_segmentados.csv",
    mime="text/csv"
)

st.markdown("---")

# ==============================================================================
# SECCIÓN 6: ANÁLISIS DE VALOR
# ==============================================================================

st.header("💎 Análisis de Índice de Valor")

# Gráfico de barras - Value Index
fig_value = px.bar(
    df_profiles.sort_values('Value_Index', ascending=False),
    x='Segmento_Nombre',
    y='Value_Index',
    color='Value_Index',
    title='Índice de Valor por Segmento (% Ingresos / % Clientes)',
    labels={'Value_Index': 'Índice de Valor', 'Segmento_Nombre': 'Segmento'},
    color_continuous_scale='RdYlGn',
    text='Value_Index'
)
fig_value.update_traces(texttemplate='%{text:.2f}x', textposition='outside')
fig_value.update_layout(showlegend=False)
fig_value.add_hline(y=1.0, line_dash="dash", line_color="red", 
                     annotation_text="Línea de equilibrio (1.0x)")

st.plotly_chart(fig_value, use_container_width=True)

st.info("""
**Índice de Valor** = (% de Ingresos) / (% de Clientes)
- **> 1.0**: Segmento genera más ingresos de lo esperado (alta eficiencia)
- **= 1.0**: Segmento balanceado
- **< 1.0**: Segmento genera menos ingresos de lo esperado (baja eficiencia)
""")

# ==============================================================================
# FOOTER
# ==============================================================================

st.markdown("---")
st.markdown("""
<div style='text-align: center; color: gray;'>
    <p>Dashboard de Segmentación RFM | Datos actualizados cada 5 minutos</p>
    <p>Powered by Databricks + Streamlit</p>
</div>
""", unsafe_allow_html=True)